# 2-2. 쌍체표본 t-검정

다이어트약 복용 전후의 체중 차이가 통계적으로 유의한지 확인한다.

- 같은 사람의 복용 전·후 데이터이므로 쌍체표본 t-검정을 사용한다.

## 이 실습의 학습 기준

이 노트북은 한 가지 함수의 결과만 확인하지 않고 다음 흐름으로 학습한다.

1. 데이터 구조에 맞는 검정 방법과 가정을 확인한다.
2. 같은 목적의 방법이 여러 개면 모두 실행해 결과를 비교한다.
3. 일부러 적합하지 않은 방법도 비교할 때는, 왜 최종 결론에 사용하지 않는지 밝힌다.
4. 전제검정 결과가 본 검정 선택에 어떻게 연결되는지 확인한다.
5. p-value뿐 아니라 차이의 방향과 크기까지 해석한다.
6. p-value가 `0.0000`으로 보이는 것은 반올림 결과이며, 실제 확률이 0이라는 뜻은 아니다.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

diet_df = pd.read_csv(
    "datas2/다이어트약_효과검증.csv",
    encoding="utf-8"
)

print(diet_df.head())
print(diet_df.columns)

   다이어트전(kg)  다이어트후(kg)
0      87.41      88.30
1      81.05      76.21
2      60.72      53.34
3      81.02      78.21
4      75.75      76.74
Index(['다이어트전(kg)', '다이어트후(kg)'], dtype='str')


## 1. 쌍체 데이터와 가설 설정

같은 사람의 다이어트 전·후 체중을 한 쌍으로 비교한다.

- 귀무가설(H₀): 다이어트 전후 평균 체중 차이는 0kg이다.
- 대립가설(H₁): 다이어트 전후 평균 체중 차이는 0kg이 아니다.

In [2]:
before = diet_df["다이어트전(kg)"]
after = diet_df["다이어트후(kg)"]

# 전 - 후가 양수이면 체중 감소를 뜻한다.
weight_difference = before - after

print("체중 차이 일부:", weight_difference.head().to_list())
print(f"평균 체중 차이: {weight_difference.mean():.2f}kg")
print("전후 자료의 쌍 개수:", len(weight_difference))

체중 차이 일부: [-0.8900000000000006, 4.840000000000003, 7.3799999999999955, 2.8100000000000023, -0.9899999999999949]
평균 체중 차이: 4.34kg
전후 자료의 쌍 개수: 50


### 1-1. 체중 차이의 KS-test와 Shapiro-Wilk 비교

쌍체표본 t-검정의 정규성 가정은 전·후 값 각각이 아니라 **각 사람의 전후 차이**에 적용한다.

- 귀무가설(H₀): 체중 차이는 정규분포를 따른다.
- 대립가설(H₁): 체중 차이는 정규분포를 따르지 않는다.

두 정규성 검정을 모두 실행하되, KS-test는 표본으로 평균과 표준편차를 추정한 학습용 비교이며 Shapiro-Wilk를 주 판단으로 사용한다. 두 독립집단의 분산을 비교하는 Levene 검정은 쌍체검정의 필수 가정이 아니므로 여기서는 사용하지 않는다.

In [3]:
difference_mean = weight_difference.mean()
difference_std = weight_difference.std(ddof=1)

difference_ks = stats.kstest(
    weight_difference,
    stats.norm.cdf,
    args=(difference_mean, difference_std)
)
difference_shapiro = stats.shapiro(weight_difference)

difference_normality_comparison = pd.DataFrame({
    "검정 방법": ["KS-test", "Shapiro-Wilk"],
    "검정통계량": [difference_ks.statistic, difference_shapiro.statistic],
    "p-value": [difference_ks.pvalue, difference_shapiro.pvalue]
})
difference_normality_comparison["결론"] = difference_normality_comparison["p-value"].apply(
    lambda p: "정규성 위반 증거 부족" if p > 0.05 else "정규성 위반 가능성"
)

display(difference_normality_comparison.round(4))
print(
    "두 검정의 결론 일치 여부:",
    (difference_ks.pvalue > 0.05) == (difference_shapiro.pvalue > 0.05)
)

,검정 방법,검정통계량,p-value,결론
0,KS-test,0.0987,0.6784,정규성 위반 증거 부족
1,Shapiro-Wilk,0.9724,0.2892,정규성 위반 증거 부족


두 검정의 결론 일치 여부: True


### 1-2. 쌍체자료를 세 가지 방식으로 비교

같은 질문을 다음 방식으로 실행해 차이를 확인한다.

- `ttest_rel(전, 후)`: 대응 관계를 직접 사용하는 올바른 쌍체표본 t-검정이다.
- `ttest_1samp(전-후, 0)`: 차이값의 평균을 0과 비교한다. 쌍체표본 t-검정과 수학적으로 같은 결과가 나와야 한다.
- `ttest_ind(전, 후)`: 전과 후를 서로 다른 사람처럼 처리한다. 대응 정보를 버리는 적합하지 않은 방법이며 비교 목적으로만 실행한다.

In [4]:
paired_result = stats.ttest_rel(before, after)
difference_one_sample_result = stats.ttest_1samp(weight_difference, popmean=0)

# 학습용 잘못된 적용: 같은 사람의 전후 값을 독립된 두 집단처럼 처리한다.
independent_for_comparison = stats.ttest_ind(
    before,
    after,
    equal_var=False
)

paired_method_comparison = pd.DataFrame({
    "방법": [
        "쌍체표본 t-test",
        "차이값 단일표본 t-test",
        "독립표본 t-test(비교용)"
    ],
    "대응 관계 반영": [True, True, False],
    "t-통계량": [
        paired_result.statistic,
        difference_one_sample_result.statistic,
        independent_for_comparison.statistic
    ],
    "자유도": [
        paired_result.df,
        difference_one_sample_result.df,
        independent_for_comparison.df
    ],
    "p-value": [
        paired_result.pvalue,
        difference_one_sample_result.pvalue,
        independent_for_comparison.pvalue
    ],
    "최종 결론 사용": [True, True, False]
})

display(paired_method_comparison.round(4))

print(
    "쌍체검정과 차이값 단일표본 검정의 t-통계량 일치:",
    np.isclose(paired_result.statistic, difference_one_sample_result.statistic)
)
print("최종 결론에는 대응 관계를 반영한 쌍체표본 t-검정을 사용한다.")

,방법,대응 관계 반영,t-통계량,자유도,p-value,최종 결론 사용
0,쌍체표본 t-test,True,9.7060,49.0000,0.000,True
1,차이값 단일표본 t-test,True,9.7060,49.0000,0.000,True
2,독립표본 t-test(비교용),False,1.9587,96.6586,0.053,False


쌍체검정과 차이값 단일표본 검정의 t-통계량 일치: True
최종 결론에는 대응 관계를 반영한 쌍체표본 t-검정을 사용한다.


### 1-3. 쌍체표본 t-검정 결과

- 전후 차이에 대한 KS-test와 Shapiro-Wilk 모두 정규성 위반의 충분한 증거를 보이지 않았다.
- 쌍체표본 t-검정과 차이값 단일표본 t-검정은 t=9.7060으로 정확히 같은 결과를 보였다.
- 대응 관계를 무시한 독립표본 t-검정은 p=0.0530으로 전혀 다른 결론을 만들었다. 이 방법은 최종 결론에 사용하지 않는다.
- 올바른 쌍체표본 t-검정의 p-value는 0.05보다 매우 작으므로 귀무가설을 기각한다.
- `전 - 후` 평균이 약 4.34kg으로 양수이므로, 이 표본에서는 평균적으로 체중이 감소했다.